## Remote Future Introspection

When the active policy is a **remote peer**, operations like
`laila.memorize` and `laila.remember` return a `RemoteFuture` — a
lightweight proxy that delegates status, result, and exception
queries back to the remote node over RPC.

The remote node runs as a **separate subprocess** so the two
policies share nothing except the WebSocket link.

This notebook demonstrates:
1. Launching a remote node as a subprocess and peering with it.
2. Triggering a `memorize` on the remote and receiving a `RemoteFuture`.
3. Introspecting the future with `laila.runtime.status()`, `.result()`, `.exception()`.
4. Triggering a `remember` on the remote and reading the recalled result.
5. Inspecting `laila.universe` — the global policy registry.

In [ ]:
import sys
import subprocess
import textwrap
import time
import laila
from laila.macros.defaults import DefaultPolicy, DefaultTCPIPProtocol

### 1 — Launch the remote node as a subprocess

The remote node starts a WebSocket server, prints its `HOST`,
`PORT`, and `SECRET` to stdout, then sleeps until terminated.

No explicit `DefaultPolicy()` or pool setup is needed — the
first `laila.*` call lazily creates a policy whose memory
already contains an alpha pool.

In [ ]:

REMOTE_SCRIPT = textwrap.dedent("""\
    import time, uuid, laila
    from laila.macros.defaults import DefaultTCPIPProtocol

    tcp = DefaultTCPIPProtocol(
        host="127.0.0.1",
        port=0,
        peer_secret_key=uuid.uuid4().hex,
    )
    laila.communication.add_connection(tcp)

    print(f"HOST={tcp.host}", flush=True)
    print(f"PORT={tcp.port}", flush=True)
    print(f"SECRET={tcp.peer_secret_key}", flush=True)
    print("READY", flush=True)

    while True:
        time.sleep(1)
""")

proc = subprocess.Popen(
    [sys.executable, "-c", REMOTE_SCRIPT],
    stdout=subprocess.PIPE,
    text=True,
)

remote_port = None
remote_secret = None
for line in proc.stdout:
    line = line.strip()
    if line.startswith("PORT="):
        remote_port = int(line.split("=", 1)[1])
    elif line.startswith("SECRET="):
        remote_secret = line.split("=", 1)[1]
    elif line == "READY":
        break

print(f"Remote subprocess started  (pid {proc.pid})")
print(f"  PORT:   {remote_port}")
print(f"  SECRET: {remote_secret}")


### 2 — Create the local node and peer with the remote

Once we call `add_tcpip_peer` with the matching secret, both
sides automatically register each other as peers.

In [ ]:
local_node = DefaultPolicy()
local_tcp  = DefaultTCPIPProtocol(
    host="127.0.0.1",
    port=0,
    peer_secret_key=remote_secret,
)

laila.active_policy = local_node
laila.communication.add_connection(local_tcp)

remote_id = laila.communication.add_tcpip_peer("127.0.0.1", remote_port, remote_secret)
time.sleep(0.3)

remote_proxy = laila.peers[remote_id]

print("Local node: ", laila.active_policy.global_id)
print("Remote peer:", remote_id)
print("Peers:      ", list(laila.peers.keys()))

### 3 — Memorize an entry on the remote node

Switch the active policy to the remote proxy.  `laila.memorize`
routes to the remote's alpha pool by default and returns a
**`RemoteFuture`** instead of a local `GroupFuture`.

In [ ]:
laila.active_policy = remote_proxy

entry = laila.constant(
    data={"message": "hello from the remote side"},
    nickname="remote-entry",
)
future = laila.memorize(entries=entry)

print("Returned type:", type(future).__name__)
print("Future id:    ", future.global_id[:60])

### 4 — Introspect the RemoteFuture

`laila.runtime` is **policy-independent** — it queries the
policy that owns the future regardless of `laila.active_policy`.

In [ ]:
print("Status (before wait):", laila.runtime.status(future))

future.wait()

print("Status (after wait): ", laila.runtime.status(future))
print("Exception:           ", laila.runtime.exception(future))
print("Python type:         ", type(future).__name__)

### 5 — Remember from the remote node and read the result

`laila.remember` on the remote proxy also returns a `RemoteFuture`.
After waiting, `laila.runtime.result()` returns the `global_id`
of the recalled entry on the remote side.

In [ ]:
recall_future = laila.remember(entry_ids=entry.global_id)

print("Recall future type:", type(recall_future).__name__)

recall_future.wait()

result_id = laila.runtime.result(recall_future)
print("Result global_id:  ", result_id)

### 6 — Inspect `laila.universe`

The universe is a unified view of all known policies — local
and remote.

In [ ]:
print("Local policies:")
for gid, pol in laila.local_policies.items():
    print(f"  {gid[:55]}  ({type(pol).__name__})")

print("\nRemote policies:")
for gid, pol in laila.remote_policies.items():
    print(f"  {gid[:55]}  ({type(pol).__name__})")

print(f"\nUniverse total: {len(laila.universe)} policies")

### 7 — Cleanup

Remove the TCP connection from the local node, then terminate
the remote subprocess.

In [ ]:
laila.active_policy = local_node
laila.communication.remove_connection(local_tcp)
print("Local connection removed.")

proc.terminate()
proc.wait(timeout=5)
print("Remote subprocess terminated.")